In [32]:
#Installing Ultralytics
!pip install ultralytics -q

In [34]:
import os
import yaml
import random
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

print("Starting Analysis...")

MODEL_PATH = None
TRAIN_DIR = None
VAL_DIR = None

# ==========================================
# 1. THE BULLETPROOF AUTO-FINDER
# ==========================================
print("🔍 Searching Kaggle directories for data...")
for root, dirs, files in os.walk('/kaggle/input'):
    
    # Find your YOLOv3 weight file
    for file in files:
        if file.endswith('.pt') and not MODEL_PATH:
            MODEL_PATH = os.path.join(root, file)
            print(f"🎯 Found Model: {MODEL_PATH}")
            
    # Find the image directories by looking for actual pictures
    if any(f.lower().endswith(('.jpg', '.png', '.jpeg')) for f in files):
        # If the folder path has 'train' in it, it's the train images
        if 'train' in root.lower() and not TRAIN_DIR:
            TRAIN_DIR = root
            print(f"🎯 Found Train Images: {TRAIN_DIR}")
            
        # If the folder path has 'val' in it, it's the val images
        elif 'val' in root.lower() and not VAL_DIR:
            VAL_DIR = root
            print(f"🎯 Found Val Images: {VAL_DIR}")

if not MODEL_PATH: raise FileNotFoundError("❌ Model .pt not found!")
if not TRAIN_DIR: raise FileNotFoundError("❌ Train images not found!")
if not VAL_DIR: raise FileNotFoundError("❌ Val images not found!")

# ==========================================
# 2. WRITE ABSOLUTE PATHS TO YAML
# ==========================================
YAML_FILE = "/kaggle/working/road_anomaly.yaml"

yaml_content = {
    "train": TRAIN_DIR,
    "val": VAL_DIR,
    "nc": 3,
    "names": {0: "Pothole", 1: "Speedbump", 2: "Crack"}
}

with open(YAML_FILE, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)
print(f"✅ YAML created using ABSOLUTE paths to bypass YOLO pathing bugs!")

# ==========================================
# 3. LOAD MODEL & REVALIDATE (Figure 4)
# ==========================================
print("\n📊 Loading YOLOv3 and running validation...")
model = YOLO(MODEL_PATH)

# Run validation to get authentic metrics
metrics = model.val(data=YAML_FILE, split="val", imgsz=640, batch=8, verbose=False)

classes = [model.names[i] for i in metrics.box.ap_class_index]
precision = metrics.box.p
recall = metrics.box.r
map50 = metrics.box.map50

# Plot Figure 4 (Per-Class Bar Chart)
x = np.arange(len(classes))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
ax.bar(x - width, precision, width, label='Precision', color='#1565C0')
ax.bar(x, recall, width, label='Recall', color='#2E7D32')
ax.bar(x + width, map50, width, label='mAP@0.5', color='#E65100')

ax.set_ylabel('Scores', fontsize=12)
ax.set_title(f'Figure 4: Per-Class Performance Metrics (LR = 0.001)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=12)
ax.set_ylim([0, 1.1])
ax.legend(loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig("/kaggle/working/Group1_Figure4_PerClass.png", dpi=300)
plt.close()
print("✅ Saved Per-Class Chart: Group1_Figure4_PerClass.png")

# ==========================================
# 4. QUALITATIVE RESULTS GRID (STRICTLY >= 2 DETECTIONS)
# ==========================================
print("\n📸 Generating Qualitative 2x3 Grid (Searching for dense scenes...)")

# Use the absolute validation directory we found in Step 1
image_dir = VAL_DIR
    
all_images = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.png'))]

# Shuffle the images so we get different ones every time we run it
random.shuffle(all_images)

dense_results = []

# Audition the images: only keep them if they have 2 or more detections
for img_path in all_images:
    result = model.predict(source=img_path, imgsz=640, conf=0.30, verbose=False)[0]
    
    if len(result.boxes) >= 2:
        dense_results.append((img_path, result))
        
    # Stop searching once we have 6 perfect images
    if len(dense_results) == 6:
        break

if len(dense_results) < 6:
    print(f"⚠️ Warning: Only found {len(dense_results)} dense images. Lower your confidence threshold!")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Figure 3: YOLOv3 Qualitative Detection Results (Dense Scenes ≥ 2 Detections)", fontsize=16, fontweight="bold", y=0.95)

for idx, (img_path, result) in enumerate(dense_results):
    annotated_img_rgb = result.plot()[..., ::-1]
    
    ax = axes[idx // 3, idx % 3]
    ax.imshow(annotated_img_rgb)
    ax.axis("off")
    ax.set_title(f"Sample {idx+1} | Detections: {len(result.boxes)}", fontsize=12, fontweight="bold")
    
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("/kaggle/working/Group1_Figure3_QualitativeGrid.png", dpi=300, bbox_inches="tight")
plt.close()
print("✅ Saved Dense Qualitative Grid: Group1_Figure3_QualitativeGrid.png")

🚀 Starting MCE 415 Analysis...
🔍 Searching Kaggle directories for data...
🎯 Found Val Images: /kaggle/input/datasets/obedhonoureje/unified-road-dataset-zip/unified_road_dataset/images/val
🎯 Found Train Images: /kaggle/input/datasets/obedhonoureje/unified-road-dataset-zip/unified_road_dataset/images/train
🎯 Found Model: /kaggle/input/models/obedhonoureje/yolov3-best-pt-lr-0-001/pytorch/default/1/best_0.001.pt
✅ YAML created using ABSOLUTE paths to bypass YOLO pathing bugs!

📊 Loading YOLOv3 and running validation...
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLOv3 summary (fused): 96 layers, 103,666,553 parameters, 0 gradients, 282.2 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.5 ms, read: 159.4±130.4 MB/s, size: 82.5 KB)
val: Scanning /kaggle/input/datasets/obedhonoureje/unified-road-dataset-zip/unified_road_dataset/labels/val... 793 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 793/793 943.1it/s 0.8s0.0s
WARNING ⚠️ val: Cache directory /ka

In [36]:
from ultralytics import YOLO
import pandas as pd

# Load your winning model
model = YOLO(MODEL_PATH)
YAML_FILE = "/kaggle/working/road_anomaly.yaml"

print("Calculating Per-Class Metrics...")
metrics = model.val(data=YAML_FILE, split="val", imgsz=640, batch=8, verbose=False)

# Format it into a clean table
table3_data = []

# Loop through the specific classes that were evaluated
for i, cls_idx in enumerate(metrics.box.ap_class_index):
    class_name = model.names[cls_idx]
    
    # Extract the specific metrics for this single class
    res = metrics.box.class_result(i)
    p, r, ap50, ap50_95 = res[0], res[1], res[2], res[3]
    
    table3_data.append({
        "Class": class_name,
        "Precision": round(float(p), 4),
        "Recall": round(float(r), 4),
        "mAP@0.5": round(float(ap50), 4)
    })

df_table3 = pd.DataFrame(table3_data)

print("\n" + "="*50)
print("TABLE 3: Per-Class Performance (LR = 0.001)")
print("="*50)
print(df_table3.to_string(index=False))
print("="*50)

Calculating Per-Class Metrics...
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLOv3 summary (fused): 96 layers, 103,666,553 parameters, 0 gradients, 282.2 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.2 ms, read: 172.9±44.3 MB/s, size: 107.6 KB)
val: Scanning /kaggle/input/datasets/obedhonoureje/unified-road-dataset-zip/unified_road_dataset/labels/val... 793 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 793/793 1.0Kit/s 0.8s<0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/obedhonoureje/unified-road-dataset-zip/unified_road_dataset/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 1.7it/s 58.1s.6ss
                   all        793       1209      0.791      0.543      0.608      0.405
Speed: 0.9ms preprocess, 69.2ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/val8

TABLE 3: P